# leaf-tensor-condition — ex2: classify tensors three ways: leaf-trainable, non-leaf, and leaf-frozen

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `leaf-tensor-condition`. Running the final beacon cell reports progress against the `Backprop: leaf tensor condition` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: leaf tensor condition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`leaf-tensor-condition`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "leaf-tensor-condition"
DD_SUBTOPIC = "Backprop: leaf tensor condition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Leaf condition — the full 3-way classification

Ex1 split tensors into LEAF vs NON-LEAF using `grad_fn is None`. Real autograd graphs have a THIRD category that matters: leaves WITHOUT `requires_grad`. They satisfy `grad_fn is None` but `.grad` never fills.

| Category               | `grad_fn`     | `requires_grad` | `.grad` after backward |
|------------------------|---------------|-----------------|------------------------|
| leaf-trainable         | `None`        | `True`          | populated              |
| non-leaf (interior)    | not-`None`    | `True`          | None (unless retained) |
| leaf-frozen (constant) | `None`        | `False`         | None — no grad needed  |

The same `(grad_fn, requires_grad)` pair determines all three. ARENA's MiniTensor uses the same 2-field condition (`recipe`, `requires_grad`).

**Why the third category matters.** Input data `x` (from a dataloader) and frozen-backbone params during fine-tuning both fall into the leaf-frozen bucket. If you accidentally set `requires_grad=True` on the data, the graph balloons and autograd keeps activations alive for backprop you never call.

### Exercise 2 — classify tensors three ways: leaf-trainable, non-leaf, and leaf-frozen

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze each tensor against the (grad_fn, requires_grad) pair to classify it as 'leaf-trainable', 'non-leaf', or 'leaf-frozen' — the full three-way split PyTorch's autograd uses.
> Keywords: leaf, requires-grad, grad-fn, autograd
> ```

**KCs targeted:** `leaf-iff-no-grad-fn`, `trainable-iff-requires-grad`

Implement `ex2_classify_three_ways(tensors)`. Take a list of `Tensor`s and return a list of strings, one per tensor:

- `'leaf-trainable'` if `grad_fn is None` AND `requires_grad` is True
- `'non-leaf'` if `grad_fn is not None`
- `'leaf-frozen'` if `grad_fn is None` AND `requires_grad` is False

Inputs:
- `tensors`: `list[Tensor]`.

Output: `list[str]` of the same length, drawn from the three labels above.

Do NOT use `.is_leaf` — implement from `grad_fn` + `requires_grad` from first principles. (`.is_leaf` returns True for BOTH leaf-trainable and leaf-frozen — it can't distinguish them on its own, which is why the 3-way split needs `requires_grad`.)

In [ ]:
def ex2_classify_three_ways(tensors):
    out = []
    for x in tensors:
        if x.grad_fn is not None:
            out.append('non-leaf')
        elif x.requires_grad:
            out.append('leaf-trainable')
        else:
            out.append('leaf-frozen')
    return out


<details><summary>Solution</summary>

```python
def ex2_classify_three_ways(tensors):
    out = []
    for x in tensors:
        if x.grad_fn is not None:
            out.append('non-leaf')
        elif x.requires_grad:
            out.append('leaf-trainable')
        else:
            out.append('leaf-frozen')
    return out
```

**Order of checks matters.** `grad_fn is not None` MUST come first. A non-leaf that happens to also have `requires_grad=True` (almost all of them) would otherwise be miscategorized as leaf-trainable. The non-leaf classification is the most specific condition.

**Why `.is_leaf` alone can't do this.** PyTorch's `.is_leaf` returns True for BOTH leaf-trainable and leaf-frozen tensors. It's a 2-way split (leaf vs interior), not a 3-way one. The 3-way classification needs the additional `requires_grad` bit to disambiguate the two leaf flavors.

**`detach()` produces a leaf-frozen.** `y.detach()` returns a new tensor that shares storage but has `grad_fn=None` and `requires_grad=False` — exactly the leaf-frozen condition. This is the standard idiom for 'freeze gradient flow through this tensor'.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()